# M-Lab Differential Measurement Notebooks

These notebooks display measures of Internet quality globally by detecting anomalous topology, routing policies
and congested interconnects between ISPs.  

See the paper [Detecting Anomalous Topology, Routing Policies, and Congested Interconnections at Internet Scale](https://arxiv.org/abs/2603.25875).   Or:
- [A short slide deck](https://docs.google.com/presentation/d/1NtlXFFAkwF5HqHWcDcrbh3TrGUdpdHKa1AWJ404kLQ4/edit?usp=sharing) — a quick introduction
<!-- - [Full slide deck](https://docs.google.com/presentation/d/1wpx45K30QD2ZIDuS5WsuRGrbcCrxs56cvuhydWRIRiw/edit?usp=sharing) — detailed methodology -->
<!-- - [Case for deploying servers](https://docs.google.com/presentation/d/1LBmas8kQtiT9Gv48fBjH3Eqbj4DXIVMGdACnTqr9K78/edit?usp=sharing) — how ISPs can use M-Lab instrumentation -->

These are all intended for public use — please share.

## [Global Metro Bar Chart](global_metro_bar_chart.ipynb)

Global interconnect quality (pain) metrics for every metro with two or more M-Lab servers.
High scores indicate asymmetric performance — where some client ISPs see different performance across different M-Lab servers.
Lower scores are better; high values suggest some users may have difficulty reaching certain local content.

Click a bar to reveal a navigation link; click the link to open Regional Details for that metro.

<details>
<summary>Documentation</summary>

### What it shows

Two metrics are plotted for each metro, computed across the top-N client ISPs
and all M-Lab sites in that metro:

**KSdistance × 10** — The maximum [Kolmogorov–Smirnov](https://en.wikipedia.org/wiki/Kolmogorov%E2%80%93Smirnov_test) distance between the
per-site CDFs for a given client ISP. It measures how differently ISP users
experience each M-Lab server. A high score means the ISP has non-uniform
connectivity to the M-Lab servers — a strong indicator of peering problems.

**Spread** — The ratio of the highest to lowest per-site geometric mean
(throughput or RTT) across M-Lab sites for a given ISP. A spread near 1.0
means all servers look the same. A high spread means some paths perform
much better or worse than others.

Low scores on both metrics are generally good.

### Controls

| Control | Effect |
|---|---|
| **Client ISPs** | Number of top-ranked ISPs (by test volume) to include |
| **Verbose** | `yes` includes single-server metros, which have no comparison data to show |

### Navigation

Click any bar to display a link below the chart. Click the link to open that
metro in the Regional Details notebook with the anchor pre-set.

</details>

## [Regional Details Dashboard](regional_details_dashboard.ipynb)

Detailed performance distributions for the cross-product of selected client ISPs
and selected M-Lab servers near an anchor metro.  
Each chart shows combined PDF and CDF for every server site, with test counts in the legend.

See the paper or slides for a discussion of how differences in these plots reveal asymmetric mid-path routing that may adversely affect users.

<details>
<summary>Documentation</summary>

### Selectors

**Anchor Metro** — Geographic centre used to pre-select servers and rank client ISPs.
Changing the anchor re-populates both Servers and Client ISPs automatically.

**Radius (km)** — How far from the anchor to look for additional M-Lab servers.

**Servers** — M-Lab sites within the selected radius. Defaults to all.

**Client Rows** — How many top-ranked ISPs to pre-process. Should be larger
than the number of ISPs you want to plot so there is headroom to select extras.

**Client ISPs** — Subset of Client Rows to include in plots. Defaults to the top half.

**Metrics** — Which performance metrics to plot: MinRTT (log and linear scale),
MeanThroughputMbps, and LossRate.

**Table** — Whether to show the summary statistics table (`none` / `Summary` / `Verbose`).

**Table Field** — Which metric the summary table reports.

### Reading the charts

Each chart row corresponds to one client ISP. Each chart shows one metric,
with one line per M-Lab server site.

The figure is split vertically: **PDF** (probability density function) in the bottom half
and **CDF** (cumulative distribution function) in the top half. Both halves share the same
x-axis and use the same colour per site. Legend labels include the total test count.

Lines that lie close together indicate uniform performance across servers.
Lines that spread apart indicate that some servers perform better or worse for some users.

### Cached data

This notebook uses pre-computed cached histograms. The actual data date range
is shown below the selectors after clicking **Run / Refresh**.

</details>

## [Fleet and Egress Load](fleet_and_egress_load.ipynb)

An internal dashboard that displays egress volume and cost across the M-Lab server fleet,
supporting fleet lifecycle management and capacity planning.

<details>
<summary>Documentation</summary>

### What it shows

**World map** — One marker per metro, coloured by log₁₀(tests/day).
Hover for metro details: servers, tests/day, TB/month, Mbps, and estimated cost.
Select `metros` and `sites` in Display to include per-site detail in hover tooltips.

**Table** — Filterable inventory at three levels:

| Level | Rows |
|---|---|
| Summaries | Global total and per-continent and per-server type subtotals |
| Metros | One row per metro with aggregate traffic and value |
| Sites | One row per M-Lab site |

Columns include server count, tests/day, average Mbps, kB/test, TB/month,
deployment SKU, and estimated daily/annual value.

### Controls

| Control | Effect |
|---|---|
| **End date** | Last day of the sample window; defaults to two days ago (UTC) |
| **Duration** | Sample window: 1, 7, or 30 days |
| **Display** | Which row levels to show in the table and map hover |

### Value model

Serving value is estimated at the flat rate of $0.10/GB egress. This generally overestimates the value in North America and Europe and underestimates the value in the Pacific rim and most of the southern Hemisphere.
Egress is measured from TCPinfo `BytesSent` for both download payload
and upload ACKs.

</details>

## [M-Lab Calibration Dashboard](m_lab_calibration_dashboard.ipynb)

Identifies potentially uncalibrated M-Lab servers by comparing measurement
distributions across sites near a target metro.
If a server is well calibrated there should be another server that yields the same results to at least one client ISP.
High scores indicate sites whose measurements differ significantly from their neighbors.
Unfortunately this version is prone to both false positive and false negative results.

**Not suitable for general use.**

<details>
<summary>Documentation</summary>

### What it shows

**Scatter plot** — KS distance vs ratio for all server pairs with ratio ≥ 1.0.
Points where the ratio exceeds 2 are clamped to the right edge and shown with
a triangle marker.

- **Ratio** — geometric mean of the alternate site ÷ geometric mean of the target
  site. A ratio > 1 means the target performs worse than the alternate.
- **KS distance** — Kolmogorov–Smirnov distance between the per-site CDFs.
  Near 0 is ideal; high values indicate distributional differences.

**Calibration report table** — Full ranked list of server pairs with columns for
KS distance, spread, ratio, test counts, and outlier flags.

### Controls

| Control | Effect |
|---|---|
| **Field** | Performance metric (MeanThroughputMbps, LossRate, …) |
| **Region** | Metro(s) to include in the search |
| **Radius (km)** | Search radius around each selected metro |
| **ISPcount** | Number of top client ISPs used to form server triplets |
| **End date** | Last day of the measurement window |
| **Duration** | Sample window: 1, 7, or 30 days |

### Interpreting results

Sites with sufficient sample sizes (both multiple comparisons and enough tests), high KS distance, and high ratio are the strongest candidates for recalibration — they differ distributionally from neighbors and show a directional performance gap.
Sites with high KS distance but ratio near 1 may have symmetric differences
that do not indicate a clear problem.

</details>

<!-- Keep this block last: experimental dashboards go at the end of the index. -->
## [Experimental Regional Details](experimental_regional_details.ipynb)

Developer version of the Regional Details dashboard with access to multiple
data source backends and additional sub-selector flags.

⚠️ **Pre-release / developer use only.** Not suitable for public use.

<details>
<summary>Documentation</summary>

For general use, select the `cached` method.

</details>